In [ ]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path('.').resolve()

MODEL_CFG = {
    'xgboost': {
        'pre_glob': REPO_ROOT / 'reports' / '08_modelo_xgboost' / '**' / 'resumen_modelos_xgboost.csv',
        'post_glob': REPO_ROOT / 'reports' / '13_tuning_xgboost' / '**' / 'resumen_tuning_xgboost.csv',
        'pre_model_col': 'modelo',
    },
    'lightgbm': {
        'pre_glob': REPO_ROOT / 'reports' / '09_modelo_lightgbm' / '**' / 'resumen_modelos_lightgbm.csv',
        'post_glob': REPO_ROOT / 'reports' / '11_tuning_lightgbm' / '**' / 'resumen_tuning_lightgbm.csv',
        'pre_model_col': 'modelo',
    },
    'redmlp': {
        'pre_glob': REPO_ROOT / 'reports' / '10_modelo_redmlp' / '**' / 'resumen_modelos_redmlp.csv',
        'post_glob': REPO_ROOT / 'reports' / '12_tuning_redmlp' / '**' / 'resumen_tuning_redmlp.csv',
        'pre_model_col': 'modelo',
    },
}

def latest_file(pattern: Path):
    files = sorted(REPO_ROOT.glob(str(pattern.relative_to(REPO_ROOT))), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None

def select_best_pre(df):
    req = ['cv_recall_target', 'cv_macro_f1']
    for c in req:
        if c not in df.columns:
            raise ValueError(f'Falta columna {c} en resumen pre-tuning')
    d = df.dropna(subset=req).copy()
    if d.empty:
        raise ValueError('Resumen pre-tuning sin filas v?lidas')
    return d.sort_values(['cv_recall_target', 'cv_macro_f1'], ascending=False).iloc[0]


In [ ]:
rows = []

for model_name, cfg in MODEL_CFG.items():
    pre_file = latest_file(cfg['pre_glob'])
    post_file = latest_file(cfg['post_glob'])

    if pre_file is None:
        print(f'[WARN] No se encontr? resumen pre-tuning para {model_name}')
        continue

    pre_df = pd.read_csv(pre_file)
    best_pre = select_best_pre(pre_df)

    pre_model = best_pre.get(cfg['pre_model_col'])
    pre_balance = best_pre.get('balanceo')

    pre_metrics = {
        'cv_recall_target_pre': best_pre.get('cv_recall_target'),
        'cv_macro_f1_pre': best_pre.get('cv_macro_f1'),
        'test_accuracy_pre': best_pre.get('accuracy_test'),
        'test_f1_macro_pre': best_pre.get('macro_f1_test'),
    }

    post_metrics = {
        'cv_recall_target_post': None,
        'cv_macro_f1_post': None,
        'test_accuracy_post': None,
        'test_f1_macro_post': None,
        'post_base_name': None,
        'post_balanceo': None,
    }

    if post_file is not None:
        post_df = pd.read_csv(post_file)
        if not post_df.empty:
            post_row = post_df.iloc[0]
            post_metrics.update({
                'cv_recall_target_post': post_row.get('cv_best_recall_target'),
                'cv_macro_f1_post': post_row.get('cv_best_f1_macro'),
                'test_accuracy_post': post_row.get('test_accuracy'),
                'test_f1_macro_post': post_row.get('test_f1_macro'),
                'post_base_name': post_row.get('base_name'),
                'post_balanceo': post_row.get('balanceo'),
            })
    else:
        print(f'[WARN] No se encontr? resumen post-tuning para {model_name}')

    row = {
        'modelo': model_name,
        'pre_file': str(pre_file),
        'post_file': str(post_file) if post_file else None,
        'pre_base_name': pre_model,
        'pre_balanceo': pre_balance,
        **pre_metrics,
        **post_metrics,
    }

    # Deltas
    for base_metric in ['cv_recall_target', 'cv_macro_f1', 'test_accuracy', 'test_f1_macro']:
        pre_v = row.get(f'{base_metric}_pre')
        post_v = row.get(f'{base_metric}_post')
        row[f'delta_{base_metric}'] = (post_v - pre_v) if pd.notna(pre_v) and pd.notna(post_v) else None

    rows.append(row)

df_cmp = pd.DataFrame(rows)
df_cmp


In [ ]:
out_dir = REPO_ROOT / 'reports' / '14_comparacion_tuning' / pd.Timestamp.today().strftime('%Y%m%d')
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / 'comparacion_pre_vs_post_tuning.csv'
df_cmp.to_csv(out_file, index=False)
print(f'Comparacion guardada en: {out_file}')
